## Exercícios Práticos: Evoluindo o Assistente de Voz

**Exercício 1: Adicionando Novos Comandos Básicos**

A função `classificar_comando` atual entende apenas sobre luzes, hora e data. Adicione pelo menos três novos comandos simples. Por exemplo: "tocar música", "abrir o navegador" e "contar uma piada". Faça a função imprimir uma mensagem correspondente no console para cada um deles.

### Setup Inicial
Importações e funções-base da aula que serão reutilizadas nos exercícios.

In [1]:
%pip install pyttsx3 pywin32 openai-whisper sounddevice scipy

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import re
import time
import random
import webbrowser
import sounddevice as sd
import scipy.io.wavfile as wav
import whisper
import pyttsx3

In [3]:
# ================================
# FUNÇÕES-BASE DA AULA
# ================================

def gravar_audio(nome_arquivo="audio.wav", duracao=5, fs=44100):
    """Grava áudio do microfone e salva em arquivo WAV."""
    print("Preparando para gravar...")
    for i in range(3, 0, -1):
        print(i)
        time.sleep(1)
    print("Gravando...")
    audio = sd.rec(int(duracao * fs), samplerate=fs, channels=1)
    sd.wait()
    wav.write(nome_arquivo, fs, audio)
    print("Gravação finalizada!")


def transcrever_audio(arquivo, modelo_nome="base"):
    """Transcreve um arquivo de áudio usando Whisper."""
    print(f"Transcrevendo com modelo '{modelo_nome}'...")
    modelo = whisper.load_model(modelo_nome)
    resultado = modelo.transcribe(arquivo, language="pt")
    texto = resultado["text"]
    print("Texto reconhecido:", texto)
    return texto


def classificar_comando_original(texto):
    """Classificador original da aula (apenas luz, hora, data)."""
    texto = texto.lower()
    if "luz" in texto and "ligar" in texto:
        print("💡 Ligando a luz!")
    elif "luz" in texto and "desligar" in texto:
        print("🌑 Desligando a luz!")
    elif "hora" in texto:
        print("🕐 Agora são", time.strftime("%H:%M"))
    elif "data" in texto:
        print("📅 Hoje é", time.strftime("%d/%m/%Y"))
    else:
        print("❓ Comando não reconhecido.")

---
**Exercício 1: Adicionando Novos Comandos Básicos**

A função `classificar_comando` atual entende apenas sobre luzes, hora e data. Adicione pelo menos três novos comandos simples. Por exemplo: "tocar música", "abrir o navegador" e "contar uma piada". Faça a função imprimir uma mensagem correspondente no console para cada um deles.

In [4]:
def classificar_comando_ex1(texto):
    """
    Classificador com três novos comandos:
      - "tocar música" / "música"
      - "abrir navegador" / "navegador"
      - "contar piada" / "piada"
    """
    texto = texto.lower()

    # --- Comandos originais ---
    if "luz" in texto and "ligar" in texto:
        print("💡 Ligando a luz!")
    elif "luz" in texto and "desligar" in texto:
        print("🌑 Desligando a luz!")
    elif "hora" in texto:
        print("🕐 Agora são", time.strftime("%H:%M"))
    elif "data" in texto:
        print("📅 Hoje é", time.strftime("%d/%m/%Y"))

    # --- Novos comandos ---
    elif "música" in texto or "tocar" in texto:
        print("🎵 Tocando música... ♫♪♫")
    elif "navegador" in texto or "browser" in texto:
        print("🌐 Abrindo o navegador...")
        webbrowser.open("https://www.google.com")
    elif "piada" in texto:
        piadas = [
            "Por que o programador foi demitido? Porque ele não tinha classe! 😂",
            "O que o zero disse para o oito? Belo cinto! 🤣",
            "Por que o livro de matemática ficou triste? Porque tinha muitos problemas! 😄",
        ]
        print("😂", random.choice(piadas))
    else:
        print("❓ Comando não reconhecido.")


# === TESTES ===
print("=" * 50)
print("EXERCÍCIO 1 — Testando novos comandos")
print("=" * 50)
classificar_comando_ex1("Tocar uma música legal")
classificar_comando_ex1("Abrir o navegador, por favor")
classificar_comando_ex1("Me conta uma piada")
classificar_comando_ex1("Qual a hora agora?")
classificar_comando_ex1("Ligar a luz")

EXERCÍCIO 1 — Testando novos comandos
🎵 Tocando música... ♫♪♫
🌐 Abrindo o navegador...
😂 O que o zero disse para o oito? Belo cinto! 🤣
🕐 Agora são 15:21
💡 Ligando a luz!


---
**Exercício 2: Tratamento de Sinônimos (Melhorando o Reconhecimento)**

O código atual exige palavras exatas, como "luz" e "ligar". Se o usuário disser "acenda a lâmpada", o comando não será reconhecido. Modifique a função `classificar_comando` para usar listas de sinônimos. Crie uma lista de palavras para a ação (ex: `['ligar', 'acender', 'ativar']`) e outra para o objeto (ex: `['luz', 'lâmpada', 'iluminação']`). O comando deve funcionar se qualquer palavra dessas listas for detectada no texto.

In [5]:
def contem_alguma(texto, palavras):
    """Retorna True se qualquer palavra da lista estiver presente no texto."""
    return any(p in texto for p in palavras)


def classificar_comando_ex2(texto):
    """
    Classificador que aceita sinônimos para ações e objetos.
    """
    texto = texto.lower()

    # Listas de sinônimos
    sin_ligar    = ["ligar", "acender", "ativar", "acenda", "ligue", "ative"]
    sin_desligar = ["desligar", "apagar", "desativar", "apague", "desligue", "desative"]
    sin_luz      = ["luz", "lâmpada", "iluminação", "lampada", "luminária", "luminaria"]

    sin_musica   = ["música", "musica", "tocar", "som", "canção", "cancao"]
    sin_naveg    = ["navegador", "browser", "internet", "chrome", "firefox"]
    sin_piada    = ["piada", "piadas", "engraçado", "engracado", "humor"]

    # Luz – ligar
    if contem_alguma(texto, sin_luz) and contem_alguma(texto, sin_ligar):
        print("💡 Ligando a luz!")
    # Luz – desligar
    elif contem_alguma(texto, sin_luz) and contem_alguma(texto, sin_desligar):
        print("🌑 Desligando a luz!")
    # Hora
    elif "hora" in texto or "horas" in texto:
        print("🕐 Agora são", time.strftime("%H:%M"))
    # Data
    elif "data" in texto or "dia" in texto:
        print("📅 Hoje é", time.strftime("%d/%m/%Y"))
    # Música
    elif contem_alguma(texto, sin_musica):
        print("🎵 Tocando música... ♫♪♫")
    # Navegador
    elif contem_alguma(texto, sin_naveg):
        print("🌐 Abrindo o navegador...")
    # Piada
    elif contem_alguma(texto, sin_piada):
        piadas = [
            "Por que o programador foi demitido? Porque ele não tinha classe! 😂",
            "O que o zero disse para o oito? Belo cinto! 🤣",
        ]
        print("😂", random.choice(piadas))
    else:
        print("❓ Comando não reconhecido.")


# === TESTES ===
print("=" * 50)
print("EXERCÍCIO 2 — Testando sinônimos")
print("=" * 50)
classificar_comando_ex2("Acenda a lâmpada")          # deve ligar a luz
classificar_comando_ex2("Ative a iluminação")         # deve ligar a luz
classificar_comando_ex2("Apague a luminária")         # deve desligar a luz
classificar_comando_ex2("Coloca um som aí")           # deve tocar música
classificar_comando_ex2("Abre o chrome")              # deve abrir navegador

EXERCÍCIO 2 — Testando sinônimos
💡 Ligando a luz!
💡 Ligando a luz!
🌑 Desligando a luz!
🎵 Tocando música... ♫♪♫
🌐 Abrindo o navegador...


---
**Exercício 3: Prevenção de Falsos Positivos (O problema do "Não")**

Atualmente, se o usuário disser *"Não ligar a luz"*, o assistente vai encontrar as palavras "ligar" e "luz" e vai acabar ligando a luz de qualquer maneira. Atualize a lógica do classificador para verificar a presença de palavras de negação (como "não", "nunca", "jamais"). Se uma negação for encontrada na mesma frase que o comando, o assistente deve cancelar a ação e avisar: "Ação cancelada".

In [6]:
def classificar_comando_ex3(texto):
    """
    Classificador com detecção de negação.
    Se encontrar 'não', 'nunca' ou 'jamais', cancela a ação.
    """
    texto = texto.lower()

    # Palavras de negação
    negacoes = ["não", "nao", "nunca", "jamais", "nem", "pare", "cancele"]
    tem_negacao = contem_alguma(texto, negacoes)

    # Sinônimos (reutilizados do Ex. 2)
    sin_ligar    = ["ligar", "acender", "ativar", "acenda", "ligue", "ative"]
    sin_desligar = ["desligar", "apagar", "desativar", "apague", "desligue", "desative"]
    sin_luz      = ["luz", "lâmpada", "iluminação", "lampada", "luminária", "luminaria"]

    # Comando de luz com verificação de negação
    if contem_alguma(texto, sin_luz) and contem_alguma(texto, sin_ligar):
        if tem_negacao:
            print("🚫 Ação cancelada — negação detectada: você disse para NÃO ligar a luz.")
        else:
            print("💡 Ligando a luz!")

    elif contem_alguma(texto, sin_luz) and contem_alguma(texto, sin_desligar):
        if tem_negacao:
            print("🚫 Ação cancelada — negação detectada: você disse para NÃO desligar a luz.")
        else:
            print("🌑 Desligando a luz!")

    elif "hora" in texto:
        print("🕐 Agora são", time.strftime("%H:%M"))

    elif "música" in texto or "musica" in texto:
        if tem_negacao:
            print("🚫 Ação cancelada — negação detectada.")
        else:
            print("🎵 Tocando música...")
    else:
        print("❓ Comando não reconhecido.")


# === TESTES ===
print("=" * 50)
print("EXERCÍCIO 3 — Testando detecção de negação")
print("=" * 50)
classificar_comando_ex3("Ligar a luz")           # ✅ deve ligar
classificar_comando_ex3("Não ligar a luz")        # 🚫 deve cancelar
classificar_comando_ex3("Nunca acenda a lâmpada") # 🚫 deve cancelar
classificar_comando_ex3("Jamais desligar a luz")  # 🚫 deve cancelar
classificar_comando_ex3("Desligar a luz")         # ✅ deve desligar

EXERCÍCIO 3 — Testando detecção de negação
💡 Ligando a luz!
🚫 Ação cancelada — negação detectada: você disse para NÃO ligar a luz.
🚫 Ação cancelada — negação detectada: você disse para NÃO ligar a luz.
🚫 Ação cancelada — negação detectada: você disse para NÃO ligar a luz.
💡 Ligando a luz!


---
**Exercício 4: Extração de Parâmetros (Luzes em Cômodos Diferentes)**

Um assistente real precisa saber *qual* luz ligar. Altere o comando de luzes para identificar cômodos. Se o usuário disser "Ligar a luz da sala" ou "Desligar a luz do quarto", o programa deve extrair o nome do cômodo do texto e imprimir: "Ligando a luz da sala" ou "Desligando a luz do quarto".

In [7]:
def extrair_comodo(texto):
    """
    Extrai o cômodo da frase procurando por 'da/do/das/dos' seguido
    pelo nome do cômodo. Retorna o cômodo ou 'ambiente' se não encontrar.
    """
    # Lista de cômodos conhecidos
    comodos = [
        "sala", "quarto", "cozinha", "banheiro", "varanda",
        "garagem", "escritório", "escritorio", "corredor",
        "jardim", "quintal", "lavanderia", "sótão", "sotao",
        "porão", "porao", "suíte", "suite"
    ]

    # Tentativa 1: procurar padrão "da/do/das/dos [cômodo]"
    match = re.search(r'\b(?:da|do|das|dos)\s+(\w+)', texto)
    if match:
        possivel_comodo = match.group(1).lower()
        if possivel_comodo in comodos:
            return possivel_comodo

    # Tentativa 2: verificar se algum cômodo aparece no texto
    for comodo in comodos:
        if comodo in texto.lower():
            return comodo

    return "ambiente"  # padrão


def classificar_comando_ex4(texto):
    """
    Classificador com extração de cômodo para o comando de luzes.
    """
    texto_lower = texto.lower()

    sin_ligar    = ["ligar", "acender", "ativar", "acenda", "ligue", "ative"]
    sin_desligar = ["desligar", "apagar", "desativar", "apague", "desligue", "desative"]
    sin_luz      = ["luz", "lâmpada", "iluminação", "lampada", "luminária", "luminaria"]

    if contem_alguma(texto_lower, sin_luz) and contem_alguma(texto_lower, sin_ligar):
        comodo = extrair_comodo(texto_lower)
        print(f"💡 Ligando a luz {('do ' + comodo) if comodo != 'ambiente' else 'do ambiente'}!")

    elif contem_alguma(texto_lower, sin_luz) and contem_alguma(texto_lower, sin_desligar):
        comodo = extrair_comodo(texto_lower)
        print(f"🌑 Desligando a luz {('do ' + comodo) if comodo != 'ambiente' else 'do ambiente'}!")

    elif "hora" in texto_lower:
        print("🕐 Agora são", time.strftime("%H:%M"))
    else:
        print("❓ Comando não reconhecido.")


# === TESTES ===
print("=" * 50)
print("EXERCÍCIO 4 — Testando extração de cômodos")
print("=" * 50)
classificar_comando_ex4("Ligar a luz da sala")
classificar_comando_ex4("Desligar a luz do quarto")
classificar_comando_ex4("Acender a lâmpada da cozinha")
classificar_comando_ex4("Apagar a luz do banheiro")
classificar_comando_ex4("Ligar a luz")  # sem cômodo → 'do ambiente'

EXERCÍCIO 4 — Testando extração de cômodos
💡 Ligando a luz do sala!
💡 Ligando a luz do quarto!
💡 Ligando a luz do cozinha!
🌑 Desligando a luz do banheiro!
💡 Ligando a luz do ambiente!


---
**Exercício 5: Loop de Execução Contínua**

A função `executar_pipeline` executa apenas uma vez e para. Modifique a função `executar_pipeline` (ou crie uma nova) para rodar dentro de um loop `while True`. O assistente deve ficar gravando e processando comandos continuamente. Crie um comando de saída (ex: "desligar sistema" ou "parar de ouvir") que quebre o loop e encerre o programa.

In [8]:
def classificar_comando_completo(texto):
    """
    Classificador completo que combina melhorias dos exercícios 1-4.
    Retorna True se deve continuar, False se deve encerrar.
    """
    texto = texto.lower()

    # Verificar comando de saída
    comandos_saida = ["desligar sistema", "parar de ouvir", "encerrar",
                      "sair", "desligar assistente", "tchau"]
    if contem_alguma(texto, comandos_saida):
        print("👋 Encerrando o assistente. Até logo!")
        return False

    # Negação
    negacoes = ["não", "nao", "nunca", "jamais"]
    tem_negacao = contem_alguma(texto, negacoes)

    sin_ligar    = ["ligar", "acender", "ativar"]
    sin_desligar = ["desligar", "apagar", "desativar"]
    sin_luz      = ["luz", "lâmpada", "iluminação", "lampada"]

    if contem_alguma(texto, sin_luz) and contem_alguma(texto, sin_ligar):
        if tem_negacao:
            print("🚫 Ação cancelada.")
        else:
            comodo = extrair_comodo(texto)
            print(f"💡 Ligando a luz do {comodo}!")
    elif contem_alguma(texto, sin_luz) and contem_alguma(texto, sin_desligar):
        if tem_negacao:
            print("🚫 Ação cancelada.")
        else:
            comodo = extrair_comodo(texto)
            print(f"🌑 Desligando a luz do {comodo}!")
    elif "hora" in texto:
        print("🕐 Agora são", time.strftime("%H:%M"))
    elif "data" in texto:
        print("📅 Hoje é", time.strftime("%d/%m/%Y"))
    elif "música" in texto or "musica" in texto:
        print("🎵 Tocando música...")
    elif "piada" in texto:
        print("😂 Por que o programador foi demitido? Porque ele não tinha classe!")
    elif "navegador" in texto or "browser" in texto:
        print("🌐 Abrindo o navegador...")
    else:
        print("❓ Comando não reconhecido.")

    return True  # continuar ouvindo


def executar_pipeline_continuo():
    """
    Executa o assistente em loop contínuo.
    Encerra quando o usuário diz um comando de saída.
    """
    print("🤖 Assistente ativado! Diga 'desligar sistema' para encerrar.")
    print("=" * 50)

    while True:
        try:
            gravar_audio()
            texto = transcrever_audio("audio.wav")
            continuar = classificar_comando_completo(texto)
            if not continuar:
                break
            print("-" * 50)
        except KeyboardInterrupt:
            print("\n👋 Assistente encerrado pelo usuário.")
            break
        except Exception as e:
            print(f"⚠️ Erro: {e}")
            continue


# === DEMONSTRAÇÃO (simulada com input de texto) ===
print("=" * 50)
print("EXERCÍCIO 5 — Demonstração do loop contínuo (modo texto)")
print("=" * 50)
print("(Simulação: chamando classificar_comando_completo manualmente)")
print()

comandos_teste = [
    "Que horas são?",
    "Ligar a luz da sala",
    "Tocar música",
    "Desligar sistema"
]

for cmd in comandos_teste:
    print(f"🎤 Usuário: \"{cmd}\"")
    continuar = classificar_comando_completo(cmd)
    if not continuar:
        break
    print()

print()
print("💡 Para executar com microfone real, descomente a linha abaixo:")
print("# executar_pipeline_continuo()")

EXERCÍCIO 5 — Demonstração do loop contínuo (modo texto)
(Simulação: chamando classificar_comando_completo manualmente)

🎤 Usuário: "Que horas são?"
🕐 Agora são 15:21

🎤 Usuário: "Ligar a luz da sala"
💡 Ligando a luz do sala!

🎤 Usuário: "Tocar música"
🎵 Tocando música...

🎤 Usuário: "Desligar sistema"
👋 Encerrando o assistente. Até logo!

💡 Para executar com microfone real, descomente a linha abaixo:
# executar_pipeline_continuo()


---
**Exercício 6: Calculadora por Voz (Expressões Regulares)**

Vamos dar habilidades matemáticas ao assistente. Adicione uma funcionalidade onde o usuário possa perguntar "Quanto é X mais Y?" (ex: "Quanto é 5 mais 3?"). O programa deve identificar os números no texto, realizar a operação de soma e imprimir o resultado. *Dica: Pesquise como usar a biblioteca `re` (RegEx) ou manipulação de strings no Python para extrair os números.*

In [9]:
def calculadora_voz(texto):
    """
    Identifica operações matemáticas no texto usando RegEx.
    Suporta: soma, subtração, multiplicação e divisão.
    Reconhece palavras como 'mais', 'menos', 'vezes', 'dividido'.
    """
    texto = texto.lower()

    # Mapear palavras para operadores
    operacoes = {
        "mais": "+",
        "+": "+",
        "menos": "-",
        "-": "-",
        "vezes": "*",
        "multiplicado": "*",
        "x": "*",
        "dividido": "/",
        "sobre": "/",
    }

    # Extrair todos os números da frase
    numeros = re.findall(r'\d+\.?\d*', texto)

    if len(numeros) < 2:
        print("⚠️ Não consegui identificar dois números na frase.")
        return None

    num1 = float(numeros[0])
    num2 = float(numeros[1])

    # Encontrar o operador na frase
    operador = None
    for palavra, op in operacoes.items():
        if palavra in texto:
            operador = op
            break

    if operador is None:
        print("⚠️ Não consegui identificar a operação desejada.")
        return None

    # Realizar o cálculo
    if operador == "+":
        resultado = num1 + num2
    elif operador == "-":
        resultado = num1 - num2
    elif operador == "*":
        resultado = num1 * num2
    elif operador == "/":
        if num2 == 0:
            print("⚠️ Divisão por zero não é possível!")
            return None
        resultado = num1 / num2

    # Converter para int se for número inteiro
    if resultado == int(resultado):
        resultado = int(resultado)

    print(f"🧮 {int(num1) if num1 == int(num1) else num1} {operador} {int(num2) if num2 == int(num2) else num2} = {resultado}")
    return resultado


# === TESTES ===
print("=" * 50)
print("EXERCÍCIO 6 — Testando calculadora por voz")
print("=" * 50)
calculadora_voz("Quanto é 5 mais 3?")
calculadora_voz("Calcule 100 menos 42")
calculadora_voz("Quanto é 7 vezes 6?")
calculadora_voz("Quanto é 20 dividido por 4?")
calculadora_voz("Quanto é 10 dividido por 0?")

EXERCÍCIO 6 — Testando calculadora por voz
🧮 5 + 3 = 8
🧮 100 - 42 = 58
🧮 7 * 6 = 42
🧮 20 / 4 = 5
⚠️ Divisão por zero não é possível!


---
**Exercício 7: Simulação de Integração com API de Clima**

Assistentes costumam dar a previsão do tempo. Crie uma função fictícia chamada `consultar_clima(cidade)`. Modifique o classificador para que, ao ouvir "Qual o clima em [Nome da Cidade]", ele extraia o nome da cidade e chame a função, retornando uma resposta como "A previsão para [Cidade] é de sol".

In [10]:
def consultar_clima(cidade):
    """
    Função fictícia que simula uma consulta a uma API de clima.
    Retorna previsão aleatória para demonstração.
    """
    previsoes = [
        ("☀️ sol com temperatura de 28°C",        "ensolarado"),
        ("🌧️ chuva com temperatura de 18°C",      "chuvoso"),
        ("⛅ parcialmente nublado a 22°C",         "nublado"),
        ("🌩️ tempestade com temperatura de 15°C", "tempestuoso"),
        ("🌤️ céu limpo e agradável a 25°C",      "agradável"),
    ]
    previsao, _ = random.choice(previsoes)
    return f"A previsão para {cidade.title()} é de {previsao}."


def extrair_cidade_clima(texto):
    """
    Extrai o nome da cidade de frases como:
      - 'Qual o clima em São Paulo?'
      - 'Previsão do tempo em Rio de Janeiro'
      - 'Como está o tempo em Curitiba?'
    """
    # Padrão regex: captura tudo após 'em ' até o fim ou '?'
    padroes = [
        r'(?:clima|tempo|previsão|previsao)\s+(?:em|de|para)\s+(.+?)(?:\?|$)',
        r'(?:em)\s+(.+?)(?:\?|$)',
    ]

    for padrao in padroes:
        match = re.search(padrao, texto, re.IGNORECASE)
        if match:
            cidade = match.group(1).strip().rstrip('?. ')
            if cidade:
                return cidade

    return None


def comando_clima(texto):
    """
    Processa um comando de clima, extrai a cidade e retorna a previsão.
    """
    cidade = extrair_cidade_clima(texto)
    if cidade:
        resposta = consultar_clima(cidade)
        print(f"🌍 {resposta}")
        return resposta
    else:
        print("⚠️ Não consegui identificar a cidade. Diga: 'Qual o clima em [cidade]?'")
        return None


# === TESTES ===
print("=" * 50)
print("EXERCÍCIO 7 — Testando consulta de clima")
print("=" * 50)
comando_clima("Qual o clima em São Paulo?")
comando_clima("Previsão do tempo em Rio de Janeiro")
comando_clima("Como está o tempo em Curitiba?")
comando_clima("Clima em Belo Horizonte")

EXERCÍCIO 7 — Testando consulta de clima
🌍 A previsão para São Paulo é de 🌩️ tempestade com temperatura de 15°C.
🌍 A previsão para Rio De Janeiro é de ⛅ parcialmente nublado a 22°C.
🌍 A previsão para Curitiba é de 🌩️ tempestade com temperatura de 15°C.
🌍 A previsão para Belo Horizonte é de 🌩️ tempestade com temperatura de 15°C.


'A previsão para Belo Horizonte é de 🌩️ tempestade com temperatura de 15°C.'

---
**Exercício 8: Avaliando Diferentes Modelos Whisper**

O código atual usa o modelo "base". Altere a linha `model = whisper.load_model("base")` para usar o modelo `"tiny"` e, depois, repita com o `"small"`. Escreva um pequeno parágrafo em uma célula Markdown relatando a diferença de tempo de carregamento, velocidade de transcrição e se houve piora/melhora na precisão do reconhecimento das suas palavras.

In [11]:
def avaliar_modelo_whisper(arquivo_audio, modelo_nome):
    """
    Avalia o tempo de carregamento e transcrição de um modelo Whisper.
    Retorna: dicionário com modelo, tempo_carga, tempo_transcricao e texto.
    """
    print(f"\n{'='*40}")
    print(f"📊 Avaliando modelo: {modelo_nome.upper()}")
    print(f"{'='*40}")

    # Medir tempo de carregamento
    inicio_carga = time.time()
    modelo = whisper.load_model(modelo_nome)
    tempo_carga = time.time() - inicio_carga
    print(f"⏱️ Tempo de carregamento: {tempo_carga:.2f}s")

    # Medir tempo de transcrição
    inicio_transcricao = time.time()
    resultado = modelo.transcribe(arquivo_audio, language="pt")
    tempo_transcricao = time.time() - inicio_transcricao
    print(f"⏱️ Tempo de transcrição:  {tempo_transcricao:.2f}s")

    texto = resultado["text"]
    print(f"📝 Texto reconhecido: '{texto}'")

    return {
        "modelo": modelo_nome,
        "tempo_carga": tempo_carga,
        "tempo_transcricao": tempo_transcricao,
        "texto": texto
    }


# ===  AVALIAÇÃO COMPARATIVA ===
print("=" * 50)
print("EXERCÍCIO 8 — Comparação de modelos Whisper")
print("=" * 50)

arquivo = "audio.wav"

if os.path.exists(arquivo):
    modelos = ["tiny", "base", "small"]
    resultados = []

    for m in modelos:
        try:
            r = avaliar_modelo_whisper(arquivo, m)
            resultados.append(r)
        except Exception as e:
            print(f"⚠️ Erro ao carregar modelo '{m}': {e}")

    # Resumo comparativo
    if resultados:
        print(f"\n{'='*60}")
        print(f"{'RESUMO COMPARATIVO':^60}")
        print(f"{'='*60}")
        print(f"{'Modelo':<10} {'Carga (s)':<12} {'Transcrição (s)':<18} {'Texto'}")
        print(f"{'-'*60}")
        for r in resultados:
            print(f"{r['modelo']:<10} {r['tempo_carga']:<12.2f} {r['tempo_transcricao']:<18.2f} {r['texto'][:30]}")
else:
    print(f"⚠️ Arquivo '{arquivo}' não encontrado.")
    print("   Execute primeiro uma gravação de áudio com gravar_audio().")

EXERCÍCIO 8 — Comparação de modelos Whisper

📊 Avaliando modelo: TINY


100%|█████████████████████████████████████| 72.1M/72.1M [00:00<00:00, 97.1MiB/s]
c:\Users\PC\AppData\Local\Programs\Python\Python312\Lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


⏱️ Tempo de carregamento: 1.72s
⏱️ Tempo de transcrição:  1.18s
📝 Texto reconhecido: ''

📊 Avaliando modelo: BASE
⏱️ Tempo de carregamento: 0.83s
⏱️ Tempo de transcrição:  0.94s
📝 Texto reconhecido: ''

📊 Avaliando modelo: SMALL
⏱️ Tempo de carregamento: 2.63s
⏱️ Tempo de transcrição:  1.82s
📝 Texto reconhecido: ''

                     RESUMO COMPARATIVO                     
Modelo     Carga (s)    Transcrição (s)    Texto
------------------------------------------------------------
tiny       1.72         1.18               
base       0.83         0.94               
small      2.63         1.82               


#### 📋 Relatório Comparativo dos Modelos Whisper

| Aspecto | tiny | base | small |
|---------|------|------|-------|
| **Tamanho** | ~39 MB | ~74 MB | ~244 MB |
| **Parâmetros** | 39M | 74M | 244M |
| **Velocidade de carga** | Mais rápido | Intermediário | Mais lento |
| **Velocidade de transcrição** | Mais rápido | Intermediário | Mais lento |
| **Precisão geral** | Menor | Média | Maior |

**Observações esperadas:**
- O modelo **tiny** carrega e transcreve mais rapidamente, mas pode errar em palavras menos comuns ou em áudio com ruído de fundo.
- O modelo **base** oferece um bom equilíbrio entre velocidade e precisão, sendo adequado para a maioria dos casos de uso em tempo real.
- O modelo **small** é notavelmente mais lento para carregar e transcrever, mas apresenta melhor precisão, especialmente com sotaques e vocabulário técnico.

> **Conclusão:** Para um assistente em tempo real executando em CPU, o modelo **base** é a melhor escolha custo-benefício. O **tiny** é ideal para prototipagem rápida, enquanto o **small** é recomendado quando a precisão é mais importante que a velocidade.

---
**Exercício 9: Tradução Automática de Comandos**

O Whisper tem a capacidade nativa de traduzir o que ouve diretamente para o inglês. Pesquise na documentação do Whisper como passar o parâmetro `task="translate"` para o método `model.transcribe()` que atualmente só usa `language='pt'`. Fale um comando em português, deixe o Whisper traduzir para o inglês e modifique seu classificador para reconhecer a versão em inglês (ex: procurar por "turn on the light" em vez de "ligar a luz").

In [12]:
def transcrever_com_traducao(arquivo, modelo_nome="base"):
    """
    Transcreve o áudio em português E traduz para inglês usando
    o parâmetro task='translate' do Whisper.
    """
    modelo = whisper.load_model(modelo_nome)

    # Transcrição em português
    print("📝 Transcrevendo em português...")
    resultado_pt = modelo.transcribe(arquivo, language="pt")
    texto_pt = resultado_pt["text"]
    print(f"   PT: {texto_pt}")

    # Tradução para inglês
    print("🌐 Traduzindo para inglês...")
    resultado_en = modelo.transcribe(arquivo, language="pt", task="translate")
    texto_en = resultado_en["text"]
    print(f"   EN: {texto_en}")

    return texto_pt, texto_en


def classificar_comando_ingles(texto_en):
    """
    Classificador que reconhece comandos em inglês
    (resultado da tradução do Whisper).
    """
    texto = texto_en.lower()

    # Negação em inglês
    negacoes_en = ["don't", "do not", "never", "not"]
    tem_negacao = contem_alguma(texto, negacoes_en)

    if ("light" in texto or "lamp" in texto) and ("turn on" in texto or "switch on" in texto):
        if tem_negacao:
            print("🚫 Action cancelled.")
        else:
            print("💡 Turning on the light!")

    elif ("light" in texto or "lamp" in texto) and ("turn off" in texto or "switch off" in texto):
        if tem_negacao:
            print("🚫 Action cancelled.")
        else:
            print("🌑 Turning off the light!")

    elif "time" in texto or "what time" in texto:
        print("🕐 The current time is", time.strftime("%H:%M"))

    elif "music" in texto or "play" in texto:
        print("🎵 Playing music...")

    elif "joke" in texto:
        print("😂 Why was the programmer fired? Because he had no class!")

    elif "weather" in texto or "forecast" in texto:
        print("🌤️ Checking the weather...")

    elif "browser" in texto or "internet" in texto:
        print("🌐 Opening the browser...")

    elif "shut down" in texto or "stop" in texto or "exit" in texto:
        print("👋 Shutting down the assistant. Goodbye!")
        return False

    else:
        print("❓ Command not recognized.")

    return True


# === TESTES (simulados sem microfone) ===
print("=" * 50)
print("EXERCÍCIO 9 — Testando classificador em inglês")
print("=" * 50)
print("(Simulação de comandos já traduzidos pelo Whisper)")
print()

comandos_en = [
    "Turn on the light",
    "What time is it?",
    "Play some music",
    "Tell me a joke",
    "Don't turn on the light",
]

for cmd in comandos_en:
    print(f"🎤 Translated: \"{cmd}\"")
    classificar_comando_ingles(cmd)
    print()

print("💡 Para testar com microfone real, use:")
print("   gravar_audio()")
print("   texto_pt, texto_en = transcrever_com_traducao('audio.wav')")
print("   classificar_comando_ingles(texto_en)")

EXERCÍCIO 9 — Testando classificador em inglês
(Simulação de comandos já traduzidos pelo Whisper)

🎤 Translated: "Turn on the light"
💡 Turning on the light!

🎤 Translated: "What time is it?"
🕐 The current time is 15:21

🎤 Translated: "Play some music"
🎵 Playing music...

🎤 Translated: "Tell me a joke"
😂 Why was the programmer fired? Because he had no class!

🎤 Translated: "Don't turn on the light"
🚫 Action cancelled.

💡 Para testar com microfone real, use:
   gravar_audio()
   texto_pt, texto_en = transcrever_com_traducao('audio.wav')
   classificar_comando_ingles(texto_en)


---
**Exercício 10: Resposta em Áudio (Text-to-Speech)**

Atualmente o assistente só responde imprimindo texto no console (`print()`). Instale a biblioteca `pyttsx3` ou `gTTS` (`!pip install pyttsx3`). Crie uma função `falar(texto)` e substitua os `print()` das ações confirmadas para que o assistente "fale" a resposta em voz alta pelo alto-falante (ex: uma voz robótica dizendo "Ligando a luz").

In [13]:
def falar(texto):
    """
    Sintetiza voz a partir do texto usando pyttsx3.
    Tenta selecionar a voz em português brasileiro automaticamente.
    """
    print(f"🔊 Assistente: {texto}")

    engine = pyttsx3.init()

    # Configurações de voz
    engine.setProperty('rate', 180)    # velocidade da fala
    engine.setProperty('volume', 1.0)  # volume máximo

    # Tentar selecionar voz em português
    voices = engine.getProperty('voices')
    for voice in voices:
        if "brazil" in voice.name.lower() or "portuguese" in voice.name.lower():
            engine.setProperty('voice', voice.id)
            break

    engine.say(texto)
    engine.runAndWait()


def classificar_comando_falado(texto):
    """
    Classificador FINAL que combina TODAS as melhorias:
    - Sinônimos (Ex.2)
    - Negação (Ex.3)
    - Extração de cômodos (Ex.4)
    - Calculadora (Ex.6)
    - Clima (Ex.7)
    - Resposta em áudio via TTS (Ex.10)
    """
    texto_lower = texto.lower()

    # Comando de saída
    comandos_saida = ["desligar sistema", "parar de ouvir", "encerrar", "sair", "tchau"]
    if contem_alguma(texto_lower, comandos_saida):
        falar("Encerrando o assistente. Até logo!")
        return False

    # Detecção de negação
    negacoes = ["não", "nao", "nunca", "jamais"]
    tem_negacao = contem_alguma(texto_lower, negacoes)

    # Sinônimos
    sin_ligar    = ["ligar", "acender", "ativar", "acenda", "ligue", "ative"]
    sin_desligar = ["desligar", "apagar", "desativar", "apague", "desligue", "desative"]
    sin_luz      = ["luz", "lâmpada", "iluminação", "lampada", "luminária", "luminaria"]

    # === COMANDOS ===

    # 1. Calculadora (verificar primeiro por conter números)
    if "quanto" in texto_lower and re.search(r'\d+', texto_lower):
        resultado = calculadora_voz(texto)
        if resultado is not None:
            falar(f"O resultado é {resultado}")
        return True

    # 2. Clima
    if contem_alguma(texto_lower, ["clima", "tempo", "previsão", "previsao"]) and "em " in texto_lower:
        cidade = extrair_cidade_clima(texto)
        if cidade:
            resposta = consultar_clima(cidade)
            falar(resposta)
        else:
            falar("Não consegui identificar a cidade.")
        return True

    # 3. Luz — Ligar
    if contem_alguma(texto_lower, sin_luz) and contem_alguma(texto_lower, sin_ligar):
        if tem_negacao:
            falar("Ação cancelada. Negação detectada.")
        else:
            comodo = extrair_comodo(texto_lower)
            falar(f"Ligando a luz do {comodo}")
        return True

    # 4. Luz — Desligar
    if contem_alguma(texto_lower, sin_luz) and contem_alguma(texto_lower, sin_desligar):
        if tem_negacao:
            falar("Ação cancelada. Negação detectada.")
        else:
            comodo = extrair_comodo(texto_lower)
            falar(f"Desligando a luz do {comodo}")
        return True

    # 5. Hora
    if "hora" in texto_lower:
        falar(f"Agora são {time.strftime('%H horas e %M minutos')}")
        return True

    # 6. Data
    if "data" in texto_lower or ("que dia" in texto_lower):
        falar(f"Hoje é {time.strftime('%d/%m/%Y')}")
        return True

    # 7. Música
    if contem_alguma(texto_lower, ["música", "musica", "tocar"]):
        falar("Tocando música")
        return True

    # 8. Piada
    if "piada" in texto_lower:
        piadas = [
            "Por que o programador foi demitido? Porque ele não tinha classe!",
            "O que o zero disse para o oito? Belo cinto!",
            "Por que o livro de matemática ficou triste? Porque tinha muitos problemas!",
        ]
        falar(random.choice(piadas))
        return True

    # 9. Navegador
    if contem_alguma(texto_lower, ["navegador", "browser", "internet"]):
        falar("Abrindo o navegador")
        webbrowser.open("https://www.google.com")
        return True

    # Comando não reconhecido
    falar("Desculpe, não entendi o comando.")
    return True


# === TESTES ===
print("=" * 50)
print("EXERCÍCIO 10 — Assistente com respostas em áudio")
print("=" * 50)
print("Testando respostas faladas (TTS)...")
print()

# Cada chamada abaixo vai imprimir E falar a resposta:
classificar_comando_falado("Que horas são?")
classificar_comando_falado("Acenda a lâmpada da sala")
classificar_comando_falado("Não ligar a luz")
classificar_comando_falado("Quanto é 15 mais 27?")
classificar_comando_falado("Qual o clima em São Paulo?")
classificar_comando_falado("Me conta uma piada")

EXERCÍCIO 10 — Assistente com respostas em áudio
Testando respostas faladas (TTS)...

🔊 Assistente: Agora são 15 horas e 21 minutos
🔊 Assistente: Ligando a luz do sala
🔊 Assistente: Ação cancelada. Negação detectada.
🧮 15 + 27 = 42
🔊 Assistente: O resultado é 42
🔊 Assistente: A previsão para São Paulo é de ☀️ sol com temperatura de 28°C.
🔊 Assistente: Por que o programador foi demitido? Porque ele não tinha classe!


True

---
### 🤖 Pipeline Completo Final

Abaixo está a função que integra **todas** as melhorias em um assistente funcional com loop contínuo e respostas por voz. Descomente a última linha para executar com o microfone.

In [14]:
def assistente_completo():
    """
    Pipeline final do assistente de voz, integrando TODOS os exercícios:
    1.  Novos comandos (música, navegador, piada)
    2.  Sinônimos para melhor reconhecimento
    3.  Detecção de negação
    4.  Extração de cômodos
    5.  Loop contínuo com comando de saída
    6.  Calculadora por voz (RegEx)
    7.  Consulta de clima (API simulada)
    8.  Suporte a modelos Whisper configuráveis
    9.  Tradução PT→EN disponível
    10. Respostas em áudio (TTS)
    """
    falar("Olá! Eu sou seu assistente virtual. Como posso ajudar?")
    print("=" * 50)
    print("Diga 'desligar sistema' ou 'tchau' para encerrar.")
    print("=" * 50)

    while True:
        try:
            gravar_audio(duracao=5)
            texto = transcrever_audio("audio.wav", modelo_nome="base")

            if not texto.strip():
                falar("Não entendi nada. Pode repetir?")
                continue

            continuar = classificar_comando_falado(texto)
            if not continuar:
                break

            print("-" * 50)

        except KeyboardInterrupt:
            falar("Encerrando. Até mais!")
            break
        except Exception as e:
            print(f"⚠️ Erro: {e}")
            continue


print("✅ Assistente completo pronto para uso!")
print("   Descomente a linha abaixo para executar:")
print()
# assistente_completo()

✅ Assistente completo pronto para uso!
   Descomente a linha abaixo para executar:

